# AG News Classification and Analysis

Code-focused workflow for loading AG News, training classifiers, and running an optional Gradio/Groq demo.


## 1. Install dependencies


## 2. Imports


In [ ]:
from data_loader import LABEL_MAP, load_ag_news
from ml_model import evaluate_ml_model, predict_category, save_model, train_ml_model


## 3. Load data


In [ ]:
train_df, test_df = load_ag_news(train_limit=12000, test_limit=2000)
train_df.head()


In [ ]:
train_df["label_name"] = train_df["label"].map(LABEL_MAP)
test_df["label_name"] = test_df["label"].map(LABEL_MAP)
train_df["label_name"].value_counts()


## 4. Train TF-IDF + Logistic Regression


In [ ]:
ml_model = train_ml_model(train_df)
ml_metrics = evaluate_ml_model(ml_model, test_df)
print(f"Accuracy: {ml_metrics['accuracy']:.3f}")
print(ml_metrics["classification_report"])


In [ ]:
save_model(ml_model, "ml_model.joblib")


## 5. Predict custom text


In [ ]:
sample_news = "The company reported strong quarterly earnings after a rise in cloud software sales."
label_id, label_name = predict_category(ml_model, sample_news)
print(label_id, label_name)


## 6. Optional deep-learning model


In [ ]:

from dl_model import evaluate_dl_model, train_dl_model

dl_model, tokenizer, x_test, history = train_dl_model(
    train_df,
    test_df,
    epochs=3,
    batch_size=128,
)
dl_metrics = evaluate_dl_model(dl_model, x_test, test_df)
print(f"Accuracy: {dl_metrics['accuracy']:.3f}")
print(dl_metrics["classification_report"])


## 7. Optional Groq analysis


In [ ]:

from rag_pipeline import NewsAnalyzer

analyzer = NewsAnalyzer(ml_model)
# print(analyzer.analyze_with_groq(sample_news))


## 8.  RAG 


In [ ]:
# Build a small vector store for retrieved examples.
# This downloads sentence-transformer weights the first time it runs.
from rag_pipeline import build_vector_store

# vector_store = build_vector_store(train_df["clean_text"].sample(2000, random_state=42))
# rag_analyzer = NewsAnalyzer(ml_model, vector_store=vector_store)
# print(rag_analyzer.analyze_with_groq(sample_news))


## 9. Gradio demo


In [ ]:
import gradio as gr

def notebook_classify_news(text, include_llm_analysis=False):
    if not text.strip():
        return "Please enter a news headline or paragraph.", ""

    _, category = predict_category(ml_model, text)
    prediction = f"Predicted category: {category}"

    if not include_llm_analysis:
        return prediction, "LLM analysis was skipped."

    try:
        return prediction, analyzer.analyze_with_groq(text)
    except Exception as exc:
        return prediction, f"LLM analysis unavailable: {exc}"

demo = gr.Interface(
    fn=notebook_classify_news,
    inputs=[
        gr.Textbox(label="News text", lines=6),
        gr.Checkbox(label="Include Groq LLM analysis", value=False),
    ],
    outputs=[gr.Textbox(label="Classification"), gr.Textbox(label="Analysis")],
    title="AG News Classification Demo",
)

demo.launch(server_name="127.0.0.1", server_port=7860, share=False)
